# Phase 4: Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned retail sales dataset into a more informative and predictive representation by creating business-driven features from the existing data.

The engineered features aim to capture temporal patterns, competition dynamics, and promotional effects that are not directly available in the raw dataset but are expected to influence daily store sales.

This notebook focuses only on feature creation and validation. Missing value handling, categorical encoding, and other preprocessing steps will be performed separately in the preprocessing phase.

### Deliverables

- Convert raw date information into meaningful temporal features.
- Engineer competition-related features.
- Engineer promotion-related features.
- Validate all newly created features.
- Produce an enriched dataset for the preprocessing phase.

## Feature Engineering Principles

Throughout this notebook, features will be created only if they satisfy one or more of the following principles:

- Capture meaningful business behavior.
- Represent temporal or seasonal patterns.
- Improve model interpretability.
- Reduce redundancy by replacing less informative raw variables.
- Avoid target leakage and future information.

In [2]:
import numpy as np
import pandas as pd

In [3]:
train_open_df=pd.read_csv("../data/processed/train_open.csv" , parse_dates=["Date"])

C:\Users\ragha\AppData\Local\Temp\ipykernel_13908\2648670108.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_open_df=pd.read_csv("../data/processed/train_open.csv" , parse_dates=["Date"])


In [4]:
train_open_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      844392 non-null  int64         
 1   DayOfWeek                  844392 non-null  int64         
 2   Date                       844392 non-null  datetime64[us]
 3   Sales                      844392 non-null  int64         
 4   Customers                  844392 non-null  int64         
 5   Open                       844392 non-null  int64         
 6   Promo                      844392 non-null  int64         
 7   StateHoliday               844392 non-null  object        
 8   SchoolHoliday              844392 non-null  int64         
 9   StoreType                  844392 non-null  str           
 10  Assortment                 844392 non-null  str           
 11  CompetitionDistance        842206 non-null  float64       
 12 

In [5]:
train_open_df["Year"]=train_open_df["Date"].dt.year
train_open_df["Month"]=train_open_df["Date"].dt.month
train_open_df["Day"]=train_open_df["Date"].dt.day
train_open_df["WeekOfYear"]=train_open_df["Date"].dt.isocalendar().week
train_open_df["Quarter"]=train_open_df["Date"].dt.quarter
train_open_df["IsMonthStart"]=(train_open_df["Date"].dt.is_month_start).astype(int)
train_open_df["IsMonthEnd"]=(train_open_df["Date"].dt.is_month_end).astype(int)
train_open_df["IsWeekend"]=(train_open_df["Date"].dt.dayofweek.isin([5,6])).astype(int)
train_open_df[["Date","Year","Month","Day","WeekOfYear","Quarter","IsMonthStart","IsMonthEnd","IsWeekend"]].head(10)

,Date,Year,Month,Day,WeekOfYear,Quarter,IsMonthStart,IsMonthEnd,IsWeekend
0,2015-07-31,2015,7,31,31,3,0,1,0
1,2015-07-31,2015,7,31,31,3,0,1,0
2,2015-07-31,2015,7,31,31,3,0,1,0
3,2015-07-31,2015,7,31,31,3,0,1,0
4,2015-07-31,2015,7,31,31,3,0,1,0
5,2015-07-31,2015,7,31,31,3,0,1,0
6,2015-07-31,2015,7,31,31,3,0,1,0
7,2015-07-31,2015,7,31,31,3,0,1,0
8,2015-07-31,2015,7,31,31,3,0,1,0
9,2015-07-31,2015,7,31,31,3,0,1,0


In [6]:
train_open_df[
    [
        "Year",
        "Month",
        "Day",
        "WeekOfYear",
        "Quarter",
        "IsMonthStart",
        "IsMonthEnd",
        "IsWeekend",
    ]
].dtypes


Year             int32
Month            int32
Day              int32
WeekOfYear      UInt32
Quarter          int32
IsMonthStart     int64
IsMonthEnd       int64
IsWeekend        int64
dtype: object

In [7]:
train_open_df[
    [
        "Year",
        "Month",
        "Day",
        "WeekOfYear",
        "Quarter",
        "IsMonthStart",
        "IsMonthEnd",
        "IsWeekend",
    ]
].isnull().sum()

Year            0
Month           0
Day             0
WeekOfYear      0
Quarter         0
IsMonthStart    0
IsMonthEnd      0
IsWeekend       0
dtype: int64

In [8]:
competition_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear"
]
train_open_df[competition_cols].info()
display(train_open_df[competition_cols].describe(include="all"))
train_open_df[competition_cols].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 3 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CompetitionDistance        842206 non-null  float64
 1   CompetitionOpenSinceMonth  575773 non-null  float64
 2   CompetitionOpenSinceYear   575773 non-null  float64
dtypes: float64(3)
memory usage: 19.3 MB


,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
count,842206.000000,575773.000000,575773.000000
mean,5457.979627,7.224879,2008.697747
std,7809.437311,3.210144,5.978048
min,20.000000,1.000000,1900.000000
25%,710.000000,4.000000,2006.000000
50%,2320.000000,8.000000,2010.000000
75%,6890.000000,10.000000,2013.000000
max,75860.000000,12.000000,2015.000000


,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
0,1270.0,9.0,2008.0
1,570.0,11.0,2007.0
2,14130.0,12.0,2006.0
3,620.0,9.0,2009.0
4,29910.0,4.0,2015.0
5,310.0,12.0,2013.0
6,24000.0,4.0,2013.0
7,7520.0,10.0,2014.0
8,2030.0,8.0,2000.0
9,3160.0,9.0,2009.0


In [9]:
train_open_df["CompetitionInfoAvailable"] = (
    train_open_df["CompetitionOpenSinceMonth"].notna() &
    train_open_df["CompetitionOpenSinceYear"].notna()
).astype(int)
train_open_df["CompetitionInfoAvailable"].value_counts()

CompetitionInfoAvailable
1    575773
0    268619
Name: count, dtype: int64

In [10]:
# train_open_df["CompetitionOpenDate"]=pd.to_datetime(train_open_df["CompetitionOpenSinceYear"].astype(str) + "-" +
#                                                     train_open_df["CompetitionOpenSinceMonth"].astype(str)+ "-01",errors="coerce")

train_open_df["CompetitionOpenDate"]=pd.to_datetime({"year": train_open_df["CompetitionOpenSinceYear"],
                                                    "month": train_open_df["CompetitionOpenSinceMonth"],
                                                    "day": 1}, errors="coerce")

train_open_df[
    [
        "CompetitionOpenSinceYear",
        "CompetitionOpenSinceMonth",
        "CompetitionOpenDate",
    ]
].head(10)

,CompetitionOpenSinceYear,CompetitionOpenSinceMonth,CompetitionOpenDate
0,2008.0,9.0,2008-09-01
1,2007.0,11.0,2007-11-01
2,2006.0,12.0,2006-12-01
3,2009.0,9.0,2009-09-01
4,2015.0,4.0,2015-04-01
5,2013.0,12.0,2013-12-01
6,2013.0,4.0,2013-04-01
7,2014.0,10.0,2014-10-01
8,2000.0,8.0,2000-08-01
9,2009.0,9.0,2009-09-01


In [11]:
train_open_df["CompetitionOpenDate"].isna().sum()

np.int64(268619)

In [12]:
display(train_open_df[train_open_df["CompetitionOpenDate"].isna()])

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Year,Month,Day,WeekOfYear,Quarter,IsMonthStart,IsMonthEnd,IsWeekend,CompetitionInfoAvailable,CompetitionOpenDate
11,12,5,2015-07-31,8959,962,1,1,0,1,a,...,2015,7,31,31,3,0,1,0,0,NaT
12,13,5,2015-07-31,8821,568,1,1,0,0,d,...,2015,7,31,31,3,0,1,0,0,NaT
15,16,5,2015-07-31,10231,979,1,1,0,1,a,...,2015,7,31,31,3,0,1,0,0,NaT
18,19,5,2015-07-31,8234,718,1,1,0,1,a,...,2015,7,31,31,3,0,1,0,0,NaT
21,22,5,2015-07-31,6566,633,1,1,0,0,a,...,2015,7,31,31,3,0,1,0,0,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844383,512,2,2013-01-01,2646,625,1,0,a,1,b,...,2013,1,1,1,1,1,0,0,0,NaT
844384,530,2,2013-01-01,2907,532,1,0,a,1,a,...,2013,1,1,1,1,1,0,0,0,NaT
844385,562,2,2013-01-01,8498,1675,1,0,a,1,b,...,2013,1,1,1,1,1,0,0,0,NaT
844389,769,2,2013-01-01,5035,1248,1,0,a,1,b,...,2013,1,1,1,1,1,0,0,0,NaT


In [13]:
train_open_df["CompetitionAgeMonths"]=((train_open_df["Date"].dt.year - train_open_df["CompetitionOpenDate"].dt.year) * 12 +
                                      (train_open_df["Date"].dt.month - train_open_df["CompetitionOpenDate"].dt.month))

In [14]:
train_open_df[
    [
        "Date",
        "CompetitionOpenDate",
        "CompetitionAgeMonths",
    ]
].head(10)

,Date,CompetitionOpenDate,CompetitionAgeMonths
0,2015-07-31,2008-09-01,82.0
1,2015-07-31,2007-11-01,92.0
2,2015-07-31,2006-12-01,103.0
3,2015-07-31,2009-09-01,70.0
4,2015-07-31,2015-04-01,3.0
5,2015-07-31,2013-12-01,19.0
6,2015-07-31,2013-04-01,27.0
7,2015-07-31,2014-10-01,9.0
8,2015-07-31,2000-08-01,179.0
9,2015-07-31,2009-09-01,70.0


In [15]:
train_open_df["CompetitionAgeMonths"].describe()

count    575773.000000
mean         60.228356
std          72.156437
min         -31.000000
25%          14.000000
50%          51.000000
75%          94.000000
max        1386.000000
Name: CompetitionAgeMonths, dtype: float64

In [16]:
train_open_df.nlargest(
    10,
    "CompetitionAgeMonths"
)[
    [
        "Store",
        "Date",
        "CompetitionOpenDate",
        "CompetitionAgeMonths"
    ]
]

,Store,Date,CompetitionOpenDate,CompetitionAgeMonths
813,815,2015-07-31,1900-01-01,1386.0
1926,815,2015-07-30,1900-01-01,1386.0
3039,815,2015-07-29,1900-01-01,1386.0
4152,815,2015-07-28,1900-01-01,1386.0
5265,815,2015-07-27,1900-01-01,1386.0
6410,815,2015-07-25,1900-01-01,1386.0
7523,815,2015-07-24,1900-01-01,1386.0
8636,815,2015-07-23,1900-01-01,1386.0
9749,815,2015-07-22,1900-01-01,1386.0
10862,815,2015-07-21,1900-01-01,1386.0


In [17]:
train_open_df["CompetitionOpenSinceYear"].value_counts().sort_index().head(10)


CompetitionOpenSinceYear
1900.0      622
1961.0      779
1990.0     3887
1994.0     1552
1995.0     1404
1998.0      766
1999.0     6213
2000.0     7631
2001.0    12157
2002.0    20736
Name: count, dtype: int64

In [18]:

train_open_df["CompetitionAgeMonths"] = train_open_df["CompetitionAgeMonths"].mask(train_open_df["CompetitionAgeMonths"] < 0, 0)


In [19]:
train_open_df.loc[
    train_open_df["CompetitionInfoAvailable"] == 0,
    "CompetitionAgeMonths"
] = -1

In [20]:
train_open_df["CompetitionAgeMonths"].describe()

count    844392.000000
mean         41.635426
std          65.395810
min          -1.000000
25%          -1.000000
50%          16.000000
75%          73.000000
max        1386.000000
Name: CompetitionAgeMonths, dtype: float64

In [21]:
promo_cols=["Promo2","Promo2SinceWeek","Promo2SinceYear","PromoInterval"]
train_open_df[promo_cols].info()
display(train_open_df[promo_cols].describe())
train_open_df[promo_cols].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Promo2           844392 non-null  int64  
 1   Promo2SinceWeek  421085 non-null  float64
 2   Promo2SinceYear  421085 non-null  float64
 3   PromoInterval    421085 non-null  str    
dtypes: float64(2), int64(1), str(1)
memory usage: 25.8 MB


,Promo2,Promo2SinceWeek,Promo2SinceYear
count,844392.000000,421085.000000,421085.000000
mean,0.498684,23.253426,2011.754019
std,0.499999,14.100569,1.660962
min,0.000000,1.000000,2009.000000
25%,0.000000,13.000000,2011.000000
50%,0.000000,22.000000,2012.000000
75%,1.000000,37.000000,2013.000000
max,1.000000,50.000000,2015.000000


,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,0,NaN,NaN,NaN
1,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,0,NaN,NaN,NaN
4,0,NaN,NaN,NaN
5,0,NaN,NaN,NaN
6,0,NaN,NaN,NaN
7,0,NaN,NaN,NaN
8,0,NaN,NaN,NaN
9,0,NaN,NaN,NaN


In [22]:
promo_mask = train_open_df["Promo2"] == 1

train_open_df.loc[promo_mask, "Promo2StartDate"] = pd.to_datetime(
    train_open_df.loc[promo_mask, "Promo2SinceYear"].astype(int).astype(str)
    + "-W"
    + train_open_df.loc[promo_mask, "Promo2SinceWeek"].astype(int).astype(str).str.zfill(2)
    + "-1",
    format="%G-W%V-%u",
    errors="coerce",
)

In [24]:
train_open_df[
    [
        "Promo2",
        "Promo2SinceWeek",
        "Promo2SinceYear",
        "Promo2StartDate",
    ]
].head(15)

,Promo2,Promo2SinceWeek,Promo2SinceYear,Promo2StartDate
0,0,NaN,NaN,NaT
1,1,13.0,2010.0,2010-03-29
2,1,14.0,2011.0,2011-04-04
3,0,NaN,NaN,NaT
4,0,NaN,NaN,NaT
5,0,NaN,NaN,NaT
6,0,NaN,NaN,NaT
7,0,NaN,NaN,NaT
8,0,NaN,NaN,NaT
9,0,NaN,NaN,NaT


In [25]:
train_open_df["Promo2StartDate"].isna().sum()

np.int64(423307)

In [28]:
train_open_df["Promo2AgeMonths"]=((train_open_df["Date"].dt.year-train_open_df["Promo2StartDate"].dt.year)*12
                                 +train_open_df["Date"].dt.month-train_open_df["Promo2StartDate"].dt.month)

In [33]:
display(train_open_df["Promo2AgeMonths"].describe())
train_open_df["Promo2AgeMonths"].isna().sum()

count    421085.000000
mean         24.981954
std          21.365954
min         -29.000000
25%           9.000000
50%          25.000000
75%          42.000000
max          72.000000
Name: Promo2AgeMonths, dtype: float64

np.int64(423307)

In [34]:
train_open_df[
    [
        "Date",
        "Promo2StartDate",
        "Promo2AgeMonths",
    ]
].head(10)

,Date,Promo2StartDate,Promo2AgeMonths
0,2015-07-31,NaT,NaN
1,2015-07-31,2010-03-29,64.0
2,2015-07-31,2011-04-04,51.0
3,2015-07-31,NaT,NaN
4,2015-07-31,NaT,NaN
5,2015-07-31,NaT,NaN
6,2015-07-31,NaT,NaN
7,2015-07-31,NaT,NaN
8,2015-07-31,NaT,NaN
9,2015-07-31,NaT,NaN


In [35]:
train_open_df["Promo2AgeMonths"]=train_open_df["Promo2AgeMonths"].mask(train_open_df["Promo2AgeMonths"]<0,0)

In [40]:
train_open_df.loc[
    train_open_df["Promo2"]==0,
    "Promo2AgeMonths"
 ] = -1

In [42]:
train_open_df["Promo2AgeMonths"].describe()

count    844392.000000
mean         12.568247
std          19.321434
min          -1.000000
25%          -1.000000
50%          -1.000000
75%          25.000000
max          72.000000
Name: Promo2AgeMonths, dtype: float64

In [43]:
train_open_df["Promo2AgeMonths"].value_counts().sort_index().head(10)

Promo2AgeMonths
-1.0    423307
 0.0     59361
 1.0      4469
 2.0      4450
 3.0      4784
 4.0      5896
 5.0      5859
 6.0      6254
 7.0      6084
 8.0      6649
Name: count, dtype: int64

In [44]:
train_open_df["TransactionMonth"] = train_open_df["Date"].dt.strftime("%b")

In [45]:
train_open_df["IsPromoMonth"] = train_open_df.apply(
    lambda row: (
        row["TransactionMonth"] in row["PromoInterval"]
        if pd.notna(row["PromoInterval"])
        else 0
    ),
    axis=1,
).astype(int)

In [46]:
train_open_df[
    [
        "Date",
        "TransactionMonth",
        "PromoInterval",
        "IsPromoMonth",
    ]
].sample(20, random_state=42)

,Date,TransactionMonth,PromoInterval,IsPromoMonth
313881,2014-08-02,Aug,NaN,0
719687,2013-05-16,May,NaN,0
95182,2015-04-20,Apr,NaN,0
8439,2015-07-23,Jul,NaN,0
455584,2014-02-24,Feb,NaN,0
530173,2013-12-04,Dec,"Jan,Apr,Jul,Oct",0
543941,2013-11-20,Nov,"Jan,Apr,Jul,Oct",0
407793,2014-04-15,Apr,"Jan,Apr,Jul,Oct",1
112910,2015-03-30,Mar,NaN,0
786239,2013-03-04,Mar,NaN,0


In [47]:
train_open_df["IsPromoMonth"].value_counts()

IsPromoMonth
0    699164
1    145228
Name: count, dtype: int64

In [48]:
engineered_cols = [
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "Quarter",
    "IsMonthStart",
    "IsMonthEnd",
    "IsWeekend",
    "CompetitionInfoAvailable",
    "CompetitionAgeMonths",
    "Promo2AgeMonths",
    "IsPromoMonth",
]

train_open_df[engineered_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Year                      844392 non-null  int32  
 1   Month                     844392 non-null  int32  
 2   Day                       844392 non-null  int32  
 3   WeekOfYear                844392 non-null  UInt32 
 4   Quarter                   844392 non-null  int32  
 5   IsMonthStart              844392 non-null  int64  
 6   IsMonthEnd                844392 non-null  int64  
 7   IsWeekend                 844392 non-null  int64  
 8   CompetitionInfoAvailable  844392 non-null  int64  
 9   CompetitionAgeMonths      844392 non-null  float64
 10  Promo2AgeMonths           844392 non-null  float64
 11  IsPromoMonth              844392 non-null  int64  
dtypes: UInt32(1), float64(2), int32(4), int64(5)
memory usage: 62.0 MB


In [49]:
train_open_df[engineered_cols].isna().sum()

Year                        0
Month                       0
Day                         0
WeekOfYear                  0
Quarter                     0
IsMonthStart                0
IsMonthEnd                  0
IsWeekend                   0
CompetitionInfoAvailable    0
CompetitionAgeMonths        0
Promo2AgeMonths             0
IsPromoMonth                0
dtype: int64

In [50]:
train_open_df[
    [
        "CompetitionAgeMonths",
        "Promo2AgeMonths",
    ]
].describe()

,CompetitionAgeMonths,Promo2AgeMonths
count,844392.000000,844392.000000
mean,41.635426,12.568247
std,65.395810,19.321434
min,-1.000000,-1.000000
25%,-1.000000,-1.000000
50%,16.000000,-1.000000
75%,73.000000,25.000000
max,1386.000000,72.000000


In [51]:
boolean_cols = [
    "IsWeekend",
    "IsMonthStart",
    "IsMonthEnd",
    "CompetitionInfoAvailable",
    "IsPromoMonth",
]

for col in boolean_cols:
    print(f"\n{col}")
    print(train_open_df[col].value_counts())


IsWeekend
IsWeekend
0    696741
1    147651
Name: count, dtype: int64

IsMonthStart
IsMonthStart
0    825024
1     19368
Name: count, dtype: int64

IsMonthEnd
IsMonthEnd
0    816351
1     28041
Name: count, dtype: int64

CompetitionInfoAvailable
CompetitionInfoAvailable
1    575773
0    268619
Name: count, dtype: int64

IsPromoMonth
IsPromoMonth
0    699164
1    145228
Name: count, dtype: int64


In [52]:
train_open_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      844392 non-null  int64         
 1   DayOfWeek                  844392 non-null  int64         
 2   Date                       844392 non-null  datetime64[us]
 3   Sales                      844392 non-null  int64         
 4   Customers                  844392 non-null  int64         
 5   Open                       844392 non-null  int64         
 6   Promo                      844392 non-null  int64         
 7   StateHoliday               844392 non-null  object        
 8   SchoolHoliday              844392 non-null  int64         
 9   StoreType                  844392 non-null  str           
 10  Assortment                 844392 non-null  str           
 11  CompetitionDistance        842206 non-null  float64       
 12 

In [54]:
train_open_df.isna().sum().sort_values(ascending=False)

Promo2SinceWeek              423307
PromoInterval                423307
Promo2StartDate              423307
PromoAgeMonths               423307
Promo2SinceYear              423307
CompetitionOpenDate          268619
CompetitionOpenSinceMonth    268619
CompetitionOpenSinceYear     268619
CompetitionDistance            2186
Store                             0
DayOfWeek                         0
StoreType                         0
StateHoliday                      0
Promo                             0
Open                              0
SchoolHoliday                     0
Date                              0
Sales                             0
Promo2                            0
Assortment                        0
Customers                         0
Year                              0
Quarter                           0
Month                             0
Day                               0
WeekOfYear                        0
IsWeekend                         0
IsMonthEnd                  

In [55]:
train_open_df.to_csv(
    "../data/processed/train_engineered.csv",
    index=False
)